# The Perceptron Learning Algorithm

Companion notebook for: [The Perceptron Learning Algorithm](https://ml-viz-ruby.vercel.app/wiki/perceptron-learning)

We:
1. Implement the perceptron from scratch in pure Python
2. Trace AND convergence and visualise weight updates
3. Demonstrate XOR failure
4. Verify the two-layer MLP fix

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a',
    'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'font.size': 11,
})

## 1 · Perceptron from scratch

In [ ]:
class Perceptron:
    def __init__(self, n_features, lr=1.0):
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.lr = lr
        self.history = []  # track (w, b) after each update

    def predict(self, x):
        return 1 if self.w @ x + self.b > 0 else 0

    def fit_step(self, x, y):
        y_hat = self.predict(x)
        error = y - y_hat
        if error != 0:
            self.w = self.w + self.lr * error * x
            self.b = self.b + self.lr * error
            self.history.append((self.w.copy(), self.b))
        return error

    def fit(self, X, y, max_epochs=100, verbose=True):
        self.history = []
        for epoch in range(max_epochs):
            errors = 0
            for xi, yi in zip(X, y):
                errors += abs(self.fit_step(xi, yi))
            if verbose:
                print(f"Epoch {epoch+1:2d}: {errors} mistake(s)  w={self.w}  b={self.b}")
            if errors == 0:
                break
        return self

## 2 · AND convergence trace

In [ ]:
X_and = np.array([[0,0],[1,0],[0,1],[1,1]], dtype=float)
y_and = np.array([0, 0, 0, 1])

print("=== AND dataset ===")
p = Perceptron(n_features=2, lr=1.0)
p.fit(X_and, y_and, max_epochs=20, verbose=True)
print(f"\nFinal weights: w={p.w}, b={p.b:.1f}")
print("Predictions:", [p.predict(x) for x in X_and], "(should be [0,0,0,1])")

## 3 · Visualise decision boundary evolution

In [ ]:
def plot_boundary(ax, w, b, title, xlim=(-0.5, 1.5)):
    colors = ['#6366f1', '#f59e0b']
    ax.scatter(X_and[:,0], X_and[:,1], c=y_and, cmap='coolwarm',
               s=200, zorder=3, edgecolors='white', lw=1.5)
    # Label points
    for xi, yi in zip(X_and, y_and):
        ax.annotate(f'({int(xi[0])},{int(xi[1])})', xi, xytext=(5,5),
                    textcoords='offset points', fontsize=9, color='#94a3b8')
    # Decision boundary: w[0]*x + w[1]*y + b = 0  →  y = -(w[0]*x + b) / w[1]
    xs = np.linspace(*xlim, 100)
    if abs(w[1]) > 1e-10:
        ys = -(w[0] * xs + b) / w[1]
        ax.plot(xs, ys, color='#34d399', lw=2)
    ax.set_xlim(*xlim); ax.set_ylim(-0.5, 1.5)
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.2)

# Show initial + a few snapshots
snapshots = [{'w': np.zeros(2), 'b': 0, 'title': 'Initial'}]
p2 = Perceptron(n_features=2, lr=1.0)

steps = [(X_and[0], y_and[0]), (X_and[1], y_and[1]),
         (X_and[2], y_and[2]), (X_and[3], y_and[3]),
         (X_and[0], y_and[0])]

for i, (xi, yi) in enumerate(steps):
    p2.fit_step(xi, yi)
    snapshots.append({'w': p2.w.copy(), 'b': p2.b,
                      'title': f'After step {i+1}: x={tuple(xi.astype(int))} y={yi}'})

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, snap in zip(axes.ravel(), snapshots):
    plot_boundary(ax, snap['w'], snap['b'], snap['title'])

plt.suptitle('Perceptron learning AND — decision boundary evolution', y=1.01)
plt.tight_layout()
plt.show()

## 4 · XOR failure

In [ ]:
X_xor = np.array([[0,0],[1,0],[0,1],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

print("=== XOR dataset (should NOT converge) ===")
p_xor = Perceptron(n_features=2, lr=1.0)
p_xor.fit(X_xor, y_xor, max_epochs=10, verbose=True)
print("\nWeights oscillate — the algorithm never converges on XOR.")

## 5 · Two-layer MLP fix

In [ ]:
def relu(z):
    return np.maximum(0, z)

# The exact weights from the wiki page
W1 = np.array([[1, 1], [1, 1]], dtype=float)
b1 = np.array([0, -1], dtype=float)
W2 = np.array([[1, -2]], dtype=float)
b2 = 0.0

print("=== MLP forward pass on all XOR inputs ===")
print(f"{'Input':>10}  {'z1':>14}  {'h (ReLU)':>14}  {'z2':>6}  {'ŷ':>4}  {'y':>4}")
print("-" * 65)

for x, y in zip(X_xor, y_xor):
    z1 = W1 @ x + b1
    h  = relu(z1)
    z2 = (W2 @ h + b2)[0]
    y_hat = int(z2 > 0)
    status = 'OK' if y_hat == y else 'FAIL'
    print(f"  {tuple(x.astype(int))!s:>8}  {tuple(z1):>14}  {tuple(h):>14}  {z2:>6.1f}  {y_hat:>4}  {y:>4}  {status}")

---

## ✏️ Your turn

### Exercise 1 — perceptron on OR

Train the perceptron on OR (output 1 when at least one input is 1). Confirm it converges. Plot the final boundary.

In [ ]:
X_or = np.array([[0,0],[1,0],[0,1],[1,1]], dtype=float)
y_or = np.array([0, 1, 1, 1])

# TODO(you): train a Perceptron on X_or, y_or and verify predictions
# p_or = Perceptron(n_features=2)
# p_or.fit(X_or, y_or)
# assert p_or.predict(np.array([0.0,0.0])) == 0
# assert p_or.predict(np.array([1.0,0.0])) == 1
print("Implement and uncomment the asserts.")

### Exercise 2 — count perceptron mistakes bound

The convergence theorem says the perceptron makes at most $R^2/\gamma^2$ mistakes, where:
- $R = \max_i \|\mathbf{x}_i\|$ (maximum feature norm)
- $\gamma$ = margin of the separator found

For AND, compute $R$ and $\gamma$ for the final weights, and check whether the actual mistake count is at most $R^2/\gamma^2$.

In [ ]:
# Final AND weights after convergence
w_final = p.w
b_final = p.b

# TODO(you):
# R = max norm of training inputs
# gamma = min_i y_i * (w . x_i + b) / ||w|| where y_i in {+1, -1}
# bound = R**2 / gamma**2
# actual_mistakes = sum of mistakes recorded during training

print("Compute R, gamma, bound, and actual_mistakes.")

<details>
<summary>Solutions</summary>

```python
# Exercise 1
p_or = Perceptron(n_features=2)
p_or.fit(X_or, y_or, verbose=False)
preds = [p_or.predict(x) for x in X_or]
assert preds == list(y_or), f"Expected {list(y_or)}, got {preds}"
print("OR: converged! predictions:", preds)

# Exercise 2
# Labels in {+1,-1} for margin calculation
y_pm = 2*y_and - 1  # {0,1} -> {-1,+1}
R = np.max(np.linalg.norm(X_and, axis=1))
w_norm = np.linalg.norm(w_final)
margins = y_pm * (X_and @ w_final + b_final) / w_norm
gamma = margins.min()
bound = R**2 / gamma**2
actual = len(p.history)  # number of weight updates = mistakes
print(f"R={R:.3f}, γ={gamma:.3f}, bound={bound:.1f}, actual={actual}")
assert actual <= bound, "Bound violated!"
print("Bound satisfied!")
```
</details>